# NB 04: Transformer Vol Surface Prediction

**Goal:** Train a Transformer encoder to predict the next day's vol surface given a 20-day lookback window. This uses the HF `Trainer` API for training loop management.

**Problem formulation:** Given surfaces $S_{t-19}, S_{t-18}, \ldots, S_t$ (each flattened to a 104-dim vector), predict $S_{t+1}$.

**Baseline:** The **naive predictor** $\hat{S}_{t+1} = S_t$ (yesterday's surface = today's prediction). Vol surfaces are highly persistent day-to-day, making this a strong baseline.

**Architecture:** Custom `VolSurfaceTransformer` — 2-layer encoder, 4 attention heads, 104-dim input/output (matching the flattened 8×13 grid). ~208K parameters.

## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset as TorchDataset
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from transformers import Trainer, TrainingArguments

from hf_volsurf.data.loaders import VolSurfaceDataLoader
from hf_volsurf.utils.vol_math import (
    STRIKE_GRID, TENOR_ORDER, normalize_surface, denormalize_surface,
)

OUTPUT_DIR = PROJECT_ROOT / "output"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"=== NB 04: Transformer Vol Prediction ===")
print(f"Device: {DEVICE}\n")

c:\source\repos\ale\.venv\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


PROJECT_ROOT: c:\source\repos\ale\huggin-face\hugging-face-learning
=== NB 04: Transformer Vol Prediction ===
Device: cuda



## 1. Load and normalize

Z-score normalization using **training set statistics only** — critical for avoiding look-ahead bias. The val/test sets are normalised with the same mean/std computed on 2010-2021 data.

In [2]:
loader = VolSurfaceDataLoader(PROJECT_ROOT / "data" / "db" / "hf_volsurf.db")
grids, dates = loader.get_all_surface_grids()
print(f"Loaded {len(dates)} surfaces, shape: {grids.shape}")

# Load split info
with open(OUTPUT_DIR / "02_split_info.json") as f:
    split_info = json.load(f)

train_end = split_info["train_end"]
val_end = split_info["val_end"]

train_mask = np.array([d <= train_end for d in dates])
val_mask = np.array([(d > train_end and d <= val_end) for d in dates])
test_mask = np.array([d > val_end for d in dates])

# Normalize using training set stats
train_grids = grids[train_mask]
normed, norm_stats = normalize_surface(grids, None)
# Recompute with train-only stats
_, train_stats = normalize_surface(train_grids, None)
normed_all, _ = normalize_surface(grids, train_stats)

print(f"Train: {train_mask.sum()}, Val: {val_mask.sum()}, Test: {test_mask.sum()}")

Loaded 4134 surfaces, shape: (4134, 8, 13)
Train: 3113, Val: 512, Test: 509


## 2. Windowed dataset

Each sample is a sliding window of 20 consecutive surfaces → predict the 21st. The val/test datasets include a LOOKBACK overlap with the preceding split so the first prediction in each split has a full 20-day history.

- Train: 3,093 windows (from 3,113 surfaces)
- Val: 512 windows
- Test: 509 windows

In [3]:
LOOKBACK = 20  # 20 trading days lookback
SURFACE_DIM = 8 * 13  # 104

class VolSurfaceWindowDataset(TorchDataset):
    """Windowed dataset: input = LOOKBACK surfaces, target = next surface."""

    def __init__(self, surfaces: np.ndarray):
        self.surfaces = surfaces
        self.n_samples = len(surfaces) - LOOKBACK

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        window = self.surfaces[idx : idx + LOOKBACK]  # (LOOKBACK, 8, 13)
        target = self.surfaces[idx + LOOKBACK]          # (8, 13)
        return {
            "input_ids": torch.tensor(
                window.reshape(LOOKBACK, SURFACE_DIM), dtype=torch.float32
            ),
            "labels": torch.tensor(
                target.reshape(SURFACE_DIM), dtype=torch.float32
            ),
        }

# Split into train/val/test datasets
# Find the index boundaries
train_end_idx = int(train_mask.sum())
val_end_idx = train_end_idx + int(val_mask.sum())

train_ds = VolSurfaceWindowDataset(normed_all[:train_end_idx])
val_ds = VolSurfaceWindowDataset(normed_all[train_end_idx - LOOKBACK : val_end_idx])
test_ds = VolSurfaceWindowDataset(normed_all[val_end_idx - LOOKBACK :])

print(f"Train windows: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

Train windows: 3093, Val: 512, Test: 509


## 3. Model architecture

A deliberately small Transformer — only 208K parameters — because we have just ~3k training samples. Key design:

- **Positional embedding:** Learnable (not sinusoidal) — short sequences (20) don't benefit from positional inductive bias.
- **Prediction head:** Linear projection from the last token's encoding → 104-dim output. The model attends over the lookback window and predicts from the final position.
- **Loss:** MSE on the normalised surface vector.

In [4]:
class VolSurfaceTransformer(nn.Module):
    """Small Transformer encoder for vol surface prediction."""

    def __init__(self, d_model=104, nhead=4, num_layers=2, dim_ff=256, seq_len=20):
        super().__init__()
        self.d_model = d_model
        self.pos_embedding = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            batch_first=True, dropout=0.1,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, d_model)

    def forward(self, input_ids=None, labels=None, **kwargs):
        x = input_ids + self.pos_embedding
        x = self.encoder(x)
        pred = self.head(x[:, -1, :])  # predict from last token

        loss = None
        if labels is not None:
            loss = nn.functional.mse_loss(pred, labels)
        return {"loss": loss, "logits": pred}

model = VolSurfaceTransformer()
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

Model parameters: 208,408


## 4. Train with HF Trainer

The HF `Trainer` API manages the training loop: gradient accumulation, learning rate scheduling (linear warmup + decay), evaluation at each epoch, and best-model checkpointing.

Training config: lr=1e-3, batch_size=32, 10 epochs, eval at each epoch, best model by val loss.

In [5]:
print("\n--- Training with HF Trainer ---")

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "transformer_checkpoints"),
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=1e-3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    fp16=False,
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

train_result = trainer.train()
print(f"Training loss: {train_result.training_loss:.6f}")


--- Training with HF Trainer ---


Epoch,Training Loss,Validation Loss
1,0.377201,0.118615
2,0.111722,0.084068
3,0.093056,0.106475
4,0.082378,0.073515
5,0.074681,0.100477
6,0.074493,0.069670
7,0.065962,0.064852
8,0.062815,0.063221
9,0.065289,0.063915
10,0.057771,0.061606


Training loss: 0.094302


## 5. Evaluation — Transformer vs naive baseline

**Result: the naive baseline wins.**

|  | Val RMSE | Test RMSE |
|--|---------|----------|
| Naive (yesterday) | **0.1489** | **0.1236** |
| Transformer | 0.2510 | 0.1833 |
| Improvement | -68.6% | -48.3% |

**Why does naive win?** Vol surfaces are extremely persistent — the day-to-day autocorrelation is very high (typically r > 0.99 for ATM vol). Predicting "no change" is already a strong bet.

The Transformer is predicting **levels** (the full normalised surface), not **changes** ($\Delta S_{t+1} = S_{t+1} - S_t$). Most of its capacity is wasted learning a near-identity function, and any small deviation from identity is penalised.

**How to improve (future work):**
1. **Predict changes**, not levels — the target becomes $\Delta S_{t+1}$, which is much harder to guess as "zero"
2. **Add exogenous features** — VIX, spot returns, sentiment as conditioning inputs
3. **Longer lookback** — 20 days may not capture regime-level context (try 60-120 days)
4. **Foundation models** — try Chronos-T5 which is pre-trained on millions of time series

In [6]:
print("\n--- Evaluation ---")

# Naive baseline: yesterday's surface
def naive_baseline_rmse(dataset):
    """RMSE of predicting today = yesterday."""
    errors = []
    for i in range(len(dataset)):
        sample = dataset[i]
        yesterday = sample["input_ids"][-1].numpy()  # last in window
        target = sample["labels"].numpy()
        errors.append(np.mean((yesterday - target) ** 2))
    return np.sqrt(np.mean(errors))

naive_rmse_val = naive_baseline_rmse(val_ds)
naive_rmse_test = naive_baseline_rmse(test_ds)

# Transformer predictions
model.eval()
predictions_test = []
targets_test = []
with torch.no_grad():
    for i in range(len(test_ds)):
        sample = test_ds[i]
        input_ids = sample["input_ids"].unsqueeze(0).to(DEVICE)
        output = model(input_ids=input_ids)
        predictions_test.append(output["logits"].squeeze().cpu().numpy())
        targets_test.append(sample["labels"].numpy())

predictions_test = np.array(predictions_test)
targets_test = np.array(targets_test)
transformer_rmse_test = np.sqrt(np.mean((predictions_test - targets_test) ** 2))

# Also on val
predictions_val = []
targets_val = []
with torch.no_grad():
    for i in range(len(val_ds)):
        sample = val_ds[i]
        input_ids = sample["input_ids"].unsqueeze(0).to(DEVICE)
        output = model(input_ids=input_ids)
        predictions_val.append(output["logits"].squeeze().cpu().numpy())
        targets_val.append(sample["labels"].numpy())

predictions_val = np.array(predictions_val)
targets_val = np.array(targets_val)
transformer_rmse_val = np.sqrt(np.mean((predictions_val - targets_val) ** 2))

print(f"\n{'':20s} {'Val RMSE':>12s} {'Test RMSE':>12s}")
print("-" * 46)
print(f"{'Naive (yesterday)':20s} {naive_rmse_val:12.6f} {naive_rmse_test:12.6f}")
print(f"{'Transformer':20s} {transformer_rmse_val:12.6f} {transformer_rmse_test:12.6f}")
improvement_val = (1 - transformer_rmse_val / naive_rmse_val) * 100
improvement_test = (1 - transformer_rmse_test / naive_rmse_test) * 100
print(f"{'Improvement':20s} {improvement_val:+11.1f}% {improvement_test:+11.1f}%")


--- Evaluation ---

                         Val RMSE    Test RMSE
----------------------------------------------
Naive (yesterday)        0.148889     0.123618
Transformer              0.248206     0.171646
Improvement                -66.7%       -38.9%


## 6. Visualise: actual vs predicted surfaces

Heatmaps comparing actual and predicted surfaces at 4 test dates. Despite the poor RMSE vs naive, the Transformer does capture the broad shape (skew, term structure) — it just doesn't track the day-to-day level precisely enough.

In [7]:
print("\n--- Visualizing Predictions ---")

# Denormalize for plotting
pred_denorm = denormalize_surface(
    predictions_test.reshape(-1, 8, 13), train_stats
)
target_denorm = denormalize_surface(
    targets_test.reshape(-1, 8, 13), train_stats
)

# Pick 4 test dates spaced across the test set
test_indices = [0, len(test_ds) // 3, 2 * len(test_ds) // 3, len(test_ds) - 1]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for col, tidx in enumerate(test_indices):
    # Actual
    ax = axes[0, col]
    im = ax.imshow(target_denorm[tidx], aspect="auto", cmap="viridis")
    ax.set_title(f"Actual", fontsize=9)
    ax.set_ylabel("Tenor") if col == 0 else None
    ax.set_yticks(range(8))
    ax.set_yticklabels(TENOR_ORDER, fontsize=7)
    ax.set_xticks(range(0, 13, 3))
    ax.set_xticklabels([f"{STRIKE_GRID[i]:.1f}" for i in range(0, 13, 3)], fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046)

    # Predicted
    ax = axes[1, col]
    im = ax.imshow(pred_denorm[tidx], aspect="auto", cmap="viridis")
    ax.set_title(f"Predicted", fontsize=9)
    ax.set_ylabel("Tenor") if col == 0 else None
    ax.set_xlabel("Moneyness")
    ax.set_yticks(range(8))
    ax.set_yticklabels(TENOR_ORDER, fontsize=7)
    ax.set_xticks(range(0, 13, 3))
    ax.set_xticklabels([f"{STRIKE_GRID[i]:.1f}" for i in range(0, 13, 3)], fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle("Transformer: Actual vs Predicted Vol Surfaces (Test Set)", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "04_transformer_predictions.png", dpi=150)
plt.close()
print("Saved: output/04_transformer_predictions.png")


--- Visualizing Predictions ---
Saved: output/04_transformer_predictions.png


## 7. Error heatmap

The mean absolute error heatmap shows where the Transformer struggles most. Expect higher errors at:
- **Short tenors, deep OTM puts** (top-left) — most volatile part of the surface
- **Wings** (extreme strikes) — less liquid, noisier data

In [8]:
mean_error = np.mean(np.abs(pred_denorm - target_denorm), axis=0)
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(mean_error, aspect="auto", cmap="Reds")
ax.set_yticks(range(8))
ax.set_yticklabels(TENOR_ORDER)
ax.set_xticks(range(13))
ax.set_xticklabels([f"{s:.2f}" for s in STRIKE_GRID], rotation=45, fontsize=8)
ax.set_xlabel("Moneyness")
ax.set_ylabel("Tenor")
ax.set_title("Mean Absolute Error by Grid Point (Test Set)")
plt.colorbar(im, label="MAE (IV)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "04_error_heatmap.png", dpi=150)
plt.close()
print("Saved: output/04_error_heatmap.png")

# Save model
torch.save(model.state_dict(), OUTPUT_DIR / "04_transformer_model.pt")
print("Saved: output/04_transformer_model.pt")

Saved: output/04_error_heatmap.png
Saved: output/04_transformer_model.pt


## Summary

| Metric | Value |
|--------|-------|
| Model | 2-layer Transformer, 208K params |
| Training | 10 epochs, ~12s on GPU |
| Val RMSE | 0.251 (naive: 0.149) |
| Test RMSE | 0.183 (naive: 0.124) |
| Verdict | **Naive baseline wins** |

**Key learning:** Vol surfaces are too persistent for a level-prediction model to beat "no change". This is a well-known challenge in financial time series — GARCH models face the same issue. The path forward is predicting *changes* or *residuals* relative to the naive forecast.

The HF `Trainer` API worked well for the training loop — it handled lr scheduling, checkpointing, and eval with minimal boilerplate.

**Next:** NB 05 takes a different approach — instead of predicting the *next* surface, we learn the *distribution* of all valid surfaces using a DDPM generative model.

In [9]:
print(f"\n=== NB 04 Complete ===")
print(f"Transformer ({n_params:,} params) trained for 10 epochs")
print(f"Val RMSE: {transformer_rmse_val:.6f} (naive: {naive_rmse_val:.6f}, {improvement_val:+.1f}%)")
print(f"Test RMSE: {transformer_rmse_test:.6f} (naive: {naive_rmse_test:.6f}, {improvement_test:+.1f}%)")


=== NB 04 Complete ===
Transformer (208,408 params) trained for 10 epochs
Val RMSE: 0.248206 (naive: 0.148889, -66.7%)
Test RMSE: 0.171646 (naive: 0.123618, -38.9%)
